# Lesson 59 · Your ticket to the GPU world — PyTorch tensors, NumPy conversion and requires_grad

**Goal**: get comfortable with the three things a `torch.Tensor` carries — dtype, shape, device — then make sense of converting to and from NumPy, and of `requires_grad`, your "GPU boarding pass".

> Compared with your hand-written `Value`: `requires_grad=True` ≈ "this node should collect gradients".

🔗 **Aurora connection**: the Month 2 dataset loader turns the numpy MFCC feature matrices produced by `aurora.audio.mfcc.mfcc()` into `torch.Tensor`s to feed the neural network; this lesson's `mel_to_batch()` is exactly that conversion layer.

← **Previous lesson** · [L58 · The training loop](L58_training_loop.ipynb)

> Last lesson covered **the training loop**: computing the loss, `optimizer.step`, convergence curves, and fitting the `make_moons` dataset.
> This lesson explores **PyTorch tensors**.

## Switching modes: from home-made autograd back to PyTorch tensors

In L54–L58 you hand-wrote a computation graph and a training loop with `Value`; from this lesson on we switch to `torch.Tensor`, mapping the same concepts onto a real framework's interface:

- `ndarray` → `Tensor`
- `requires_grad` → autograd
- `device` → CPU / GPU

This isn't a change of subject — it's carrying the same way of thinking over into production-grade tools.

## This lesson's story: why does NumPy get "stopped at the GPU's door"?

You've already implemented MFCC, the FFT and the mel filterbank with NumPy — they run perfectly on the CPU. But when you feed them to a PyTorch neural network, you hit a hurdle:

**NumPy doesn't know what a "gradient" is, and it can't live on the GPU.**

PyTorch's `Tensor` is an upgraded ndarray with three extra weapons:

| Capability | NumPy ndarray | PyTorch Tensor |
|---|---|---|
| Stores numbers | ✅ | ✅ |
| GPU acceleration | ❌ | ✅ (`device='cuda'`) |
| Automatic gradients | ❌ | ✅ (`requires_grad=True`) |
| Converting to/from NumPy | — | `from_numpy` / `.numpy()` |

**The key conversions**:
```python
arr = np.array([1.0, 2.0, 3.0])          # NumPy float64
t = torch.from_numpy(arr)                  # zero-copy, shared memory!
arr[0] = 99.0                              # change arr → t changes too!
t2 = torch.tensor(arr, dtype=torch.float32)  # a copy, float32
```

**Aurora connection**: `mel_to_batch()` turns a list of NumPy MFCCs into a `float32` Tensor of shape `(B, 1, T, n_mels)`, the standard input format for a CNN (batch, channel, height, width).

Your task in this lesson: understand the three main Tensor attributes (dtype/shape/device) and implement `mel_to_batch()`.

In [ ]:
import numpy as np  # numpy is always available (the only hard dependency of the Aurora Audio Core)

try:
    import torch
except ImportError as e:
    # torch is an optional dependency of Aurora (the speech/music/llm extras in pyproject.toml);
    # the Audio Core stays numpy-only, so the PyTorch lessons from here on need it installed separately.
    raise ImportError(
        "This lesson needs PyTorch: pip install torch (or pip install -e '.[speech]')"
    ) from e

print('torch', torch.__version__, '| numpy', np.__version__)

## 1. Tensor vs ndarray: dtype · shape · device

| Attribute | numpy | torch |
|------|-------|-------|
| Type | `arr.dtype` | `t.dtype` (`torch.float32` and so on) |
| Shape | `arr.shape` → tuple | `t.shape` → `torch.Size` |
| Device | always the CPU | `t.device`: `cpu` or `cuda:0` |

**dtype in depth: why does NumPy default to float64 while PyTorch defaults to float32?**

You may have noticed that `np.array([1.0])` is `float64`, but `torch.tensor([1.0])` is `float32`. That's no coincidence — there's hardware design behind it:

- **float64 (double precision)**: 8 bytes per number, high precision, but on a CPU it runs at roughly the same speed as float32; NumPy is aimed at scientific computing and would rather be slow than imprecise.
- **float32 (single precision)**: 4 bytes per number, precise enough, and on a GPU **2–8 times** as fast as float64. Why so much faster?
  - When GPU chips are designed, they get more single-precision hardware units than double-precision ones (more float32 compute units fit in the same chip area).
  - Memory bandwidth: float32 moves half as much data per transfer as float64, cutting I/O waits.
  - When matrices are large (millions of parameters), memory itself becomes the bottleneck; float32 lets bigger models fit into GPU memory.

For deep learning, float32 precision is plenty: changes in the loss function are usually on the order of 1e-3, and float32 is precise to about 1e-7. **Extra precision just wastes the GPU**. That's why modern deep-learning frameworks all default to float32.

Conversion: `torch.from_numpy(arr)` shares memory (zero-copy), and `t.numpy()` converts back (CPU tensors only).

In [ ]:
arr = np.array([[1.0, 2.0], [3.0, 4.0]])   # float64
t   = torch.from_numpy(arr)                 # shared memory, dtype=float64
t32 = t.float()                             # convert to float32

print('numpy dtype :', arr.dtype)
print('torch dtype :', t.dtype, '->', t32.dtype)
print('shape        :', t.shape)
print('device       :', t.device)

# changing arr affects t (shared memory)
arr[0, 0] = 99.0
print('t[0,0] after arr mutation:', t[0, 0].item())  # 99.0

In [ ]:
# In depth: memory and speed of float32 vs float64
print("=== float32 vs float64 compared ===")
t64 = torch.zeros(1000, dtype=torch.float64)
t32 = torch.zeros(1000, dtype=torch.float32)

print(f"float64: {t64.element_size()} bytes/element")
print(f"float32: {t32.element_size()} bytes/element")
print(f"memory saving: float32 is {t64.element_size() / t32.element_size():.0f}x smaller than float64")
print(f"on a GPU, float32 is usually 2-8x faster than float64 (more compute units fit on the same chip)")

# Precision compared: float32 is precise to 1e-7, plenty for gradient changes in a neural network (usually on the order of 1e-3)
print(f"\nfloat32 smallest precision unit: ~1e-7")
print(f"typical size of gradient changes in a neural network: ~1e-3 to 1e-4")
print(f"conclusion: float32 precision is enough for deep learning; float64 wastes the GPU")

In [ ]:
# In depth: the mechanics of zero-copy and memory sharing in torch.from_numpy()
print("\n=== Zero-copy vs copy: what happens underneath ===")

# Method 1: from_numpy (zero-copy, shared memory)
arr_shared = np.array([1.0, 2.0, 3.0])
t_shared = torch.from_numpy(arr_shared)

print(f"Method 1: from_numpy (zero-copy, shared memory)")
print(f"  before changing arr: arr={arr_shared}, t={t_shared}")
arr_shared[0] = 99.0
print(f"  after changing arr: arr={arr_shared}, t={t_shared}  ← both changed!")
print(f"  object ids: arr={id(arr_shared)}, t={id(t_shared)}  ← different objects")
print(f"  but they point to the same underlying buffer ← the secret of zero-copy")

# Method 2: torch.tensor (a copy, separate memory)
arr_copy = np.array([1.0, 2.0, 3.0])
t_copy = torch.tensor(arr_copy)

print(f"\nMethod 2: torch.tensor() (a copy)")
print(f"  before changing arr: arr={arr_copy}, t={t_copy}")
arr_copy[0] = 99.0
print(f"  after changing arr: arr={arr_copy}, t={t_copy}  ← only arr changed, t stays the same")
print(f"  why: torch.tensor() copies the numpy data into a new buffer, so the two are completely independent")

# When to use which?
print(f"\nWhen to use from_numpy vs torch.tensor?")
print(f"• from_numpy: large arrays (saves memory), when you won't modify the original ndarray")
print(f"• torch.tensor: when you need to modify the ndarray, or change the dtype (converted automatically)")

# torch.tensor() can change the dtype for you, but from_numpy keeps the original dtype
print(f"\ndtype conversion demo:")
arr_f64 = np.array([1.0, 2.0], dtype=np.float64)
t_from = torch.from_numpy(arr_f64)
t_copy = torch.tensor(arr_f64, dtype=torch.float32)  # converted to float32
print(f"  from_numpy keeps float64: {t_from.dtype}")
print(f"  torch.tensor can be told float32: {t_copy.dtype}")

## 2. Automatic gradient tracking: `requires_grad` and the computation graph

**What is a computation graph?**

When you run operations on a Tensor with `requires_grad=True`, PyTorch doesn't just compute the result and throw the rest away — it keeps a record, like writing a diary:
```
x = tensor(3.0, requires_grad=True)
y = x**2                    ← records the operation "square"
z = y + 2*x                 ← records the operations "add and multiply"
```

This forms a **directed acyclic graph (DAG)**:
```
      x (requires_grad=True)
      ├─ mul(2) ──┐
      │           ├─ add ──→ z
      └─ pow(2) ──┘
```

**Why is the DAG useful?** When you call `z.backward()`, PyTorch flows **backwards** along the DAG and differentiates with the chain rule:
```
dz/dx = dz/dy · dy/dx + dz/d(2x) · d(2x)/dx
      = 1 · 2x + 1 · 2
      = 2x + 2
```
At `x=3.0`, `dz/dx = 8.0`.

**Why can only a scalar (a single number) call `.backward()`?**

`.backward()` needs a scalar output, because a gradient is, by definition, the derivative of a scalar with respect to the inputs. Say `y.shape=(3,4)` (12 numbers) and you want the derivative with respect to `x` — which number's derivative? The derivative of "all 12 added together"? Of "the first one"? **It's ambiguous**.

The fixes:
```python
# Wrong: y is a vector, can't differentiate
y = x**2  # shape (3,4)
y.backward()  # RuntimeError!

# Right, method 1: sum it into a scalar
y.sum().backward()

# Right, method 2: pass an explicit gradient vector (same shape as y)
y.backward(torch.ones_like(y))  # equivalent to y.sum()
```

Both methods first "project" everything onto a single number, then backpropagate the gradient. PyTorch's convention: only a scalar can call `.backward()` on its own.

**Gradient accumulation**

Only leaf nodes (created by the user, not the result of an operation) accumulate `.grad`; intermediate nodes don't keep their gradients by default (`retain_grad` keeps them).
More importantly: after each `.backward()`, `.grad` is **not reset** — it accumulates:
```python
x.grad += the newly computed gradient
```
This is a trap for beginners (see experiment B), but in some situations (mini-batch gradient accumulation) it's essential.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x          # y = 9 + 6 = 15
y.backward()            # dy/dx = 2x+2 = 8

print('y      =', y.item())       # 15.0
print('x.grad =', x.grad.item()) # 8.0  ← matches the hand calculation

In [ ]:
# In depth: inspect the structure of the computation graph (DAG)
print("=== The structure of the computation graph (DAG) ===")
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
z = y + 2 * x

print(f"how z was computed: {z.grad_fn}")  # print a DAG node
print(f"  ↓ the two inputs of z = add(y, 2*x): {z.grad_fn.next_functions}")
print("when you call z.backward(), the gradient flows backwards along this chain")

# backward() on a non-scalar tensor needs special handling
print("\n=== A non-scalar tensor must become a scalar first ===")
x_vec = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y_vec = x_vec ** 2  # shape (3,)
print(f"y_vec.shape = {y_vec.shape} (not a scalar)")

# Wrong example (commented out, otherwise the cell would stop here)
# y_vec.backward()  # RuntimeError: grad can only be implicitly created for scalar outputs

# Right, method 1: sum
y_vec.sum().backward()
print(f"after y_vec.sum().backward(), x_vec.grad = {x_vec.grad}")  # [2, 4, 6]

# Reset the gradient and try method 2
x_vec.grad = None
y_vec = x_vec ** 2
y_vec.backward(torch.ones(3))  # pass the gradient explicitly
print(f"after y_vec.backward(torch.ones(3)), x_vec.grad = {x_vec.grad}")  # also [2, 4, 6]

## 3. Common tensor operations

**Reshaping: view vs reshape**

- `view(shape)` / `reshape(shape)`: rearrange the dimensions, keeping the total number of elements. `view` needs contiguous memory; `reshape` handles it for you.

  **What is contiguous memory?** A Tensor is stored in memory as one contiguous 1-D run, and `view` needs that 1-D ordering to match the "traversal order" of the new shape. For example, going from `(2,3)` to `(3,2)`, the order in physical memory has to line up. Some operations (such as the transpose `.T`) break contiguity — then `view` raises an error, and you must use `reshape` or fix it first with `.contiguous()`.

  ```python
  t = torch.arange(6).reshape(2, 3)    # (2, 3)
  t_T = t.T                             # (3, 2), memory not contiguous!
  # t_T.view(-1) fails, because the physical memory order is [0,1,2,3,4,5] but after the transpose the logical order should be [0,3,1,4,2,5]

  # fix 1: use reshape (handled automatically)
  t_T.reshape(-1)  # works

  # fix 2: make it contiguous first, then view
  t_T.contiguous().view(-1)
  ```

- `squeeze(dim)` / `unsqueeze(dim)`: remove / insert a dimension of size 1.

**Joining: cat vs stack**

These two operations look alike, but they do quite different jobs:

- `torch.cat([a, b], dim=0)`: **joins along an existing dimension**. The two tensors must match in every dimension except the one being joined.
  ```python
  x: (2, 3)
  y: (2, 3)
  cat(x, y, dim=0) → (4, 3)  # rows stacked on rows, no new dimension
  ```

- `torch.stack([a, b], dim=0)`: **creates a new dimension, then joins**. All the tensors must have exactly the same shape.
  ```python
  x: (2, 3)
  y: (2, 3)
  stack(x, y, dim=0) → (2, 2, 3)  # a new dimension inserted at the front
  ```

**When to use which?**

- **cat**: lengthen the same dimension of the same kind of data. For example, you read 10 batches, all of `(32, 224, 224)` images, and want to merge them into `(320, 224, 224)`.
- **stack**: pack separate samples into a batch. For example, B different samples' MFCC features, each `(T, 40)`, need packing into a CNN input: first `stack` them into `(B, T, 40)`, then `unsqueeze(1)` to get `(B, 1, T, 40)`.

**Multiplication**

- `@` (same as `torch.matmul`): matrix multiplication, with batch broadcasting.
- `torch.einsum('bi,ij->bj', x, W)`: Einstein summation (einsum) — writing out the indices is clearer than repeated `transpose`s.

**The full einsum rules:**

Einstein summation uses letters for dimensions, and a repeated letter means summing over it (contraction):
```
torch.einsum('bi,ij->bj', x, W)
           ↓  ↓  ↓
         input1 input2 → output

Rules:
1. each letter is the index of one dimension
2. a repeated letter (here i) means summing over that dimension
3. the right side of the arrow lists only the indices that are "kept", in the order of the output dimensions

Quick reference:
- matrix multiplication: 'ij,jk->ik' (j repeated → sum over j)
- transpose: 'ij->ji' (swap the output order)
- batched matrix multiplication: 'bij,bjk->bik' (keep b, sum over j)
- matrix–vector product: 'ij,j->i' (sum over j, keep i)
- vector dot product: 'i,i->' (sum over i, no output dimensions = a scalar)
```

In [ ]:
# --- view / unsqueeze / memory contiguity ---
print("=== Memory contiguity: view vs reshape ===")
a = torch.arange(12).float().reshape(3, 4)
print(f"original a: shape {a.shape}")
print(f"a.is_contiguous() = {a.is_contiguous()}")

a_T = a.T  # transpose, breaking contiguity
print(f"\nafter transposing, a_T: shape {a_T.shape}")
print(f"a_T.is_contiguous() = {a_T.is_contiguous()}")

# view raises an error, reshape copes
try:
    a_T.view(-1)
except RuntimeError as e:
    print(f"a_T.view(-1) failed: {e}")

# fix 1: use reshape (handled automatically)
print(f"a_T.reshape(-1) works: {a_T.reshape(-1).shape}")

# fix 2: contiguous() first, then view
print(f"a_T.contiguous().view(-1) works: {a_T.contiguous().view(-1).shape}")

# --- cat vs stack ---
print("\n=== cat vs stack: when to use which ===")
x = torch.ones(2, 3)
y = torch.zeros(2, 3)

cat_result = torch.cat([x, y], dim=0)
stack_result = torch.stack([x, y], dim=0)

print(f"x.shape: {x.shape}, y.shape: {y.shape}")
print(f"cat([x,y], dim=0) → {cat_result.shape}  # rows stacked on rows, no new dimension")
print(f"stack([x,y], dim=0) → {stack_result.shape}  # new dimension at the front")

# A real scenario: packing MFCC features (why stack)
print("\n[Scenario] Why use stack to pack MFCC features?")
mel_list = [torch.randn(80, 40) for _ in range(3)]  # 3 samples
batch_stacked = torch.stack(mel_list)  # → (3, 80, 40)
print(f"3 MFCCs of (80,40) after stack: {batch_stacked.shape}")
print("✓ use stack because each sample is separate and needs packing along a new batch dimension")

# --- einsum in depth ---
print("\n=== einsum explained in detail ===")

# Example 1: matrix multiplication (n, m) × (m, k) → (n, k)
A = torch.randn(4, 3)  # n=4, m=3
B = torch.randn(3, 5)  # m=3, k=5
result_matmul = A @ B
result_einsum = torch.einsum('nm,mk->nk', A, B)
print(f"matrix multiplication: A({A.shape}) @ B({B.shape}) = {result_matmul.shape}")
print(f"  einsum('nm,mk->nk', A, B) gives {result_einsum.shape}")
print(f"  check that they are equal: {torch.allclose(result_matmul, result_einsum)}")

# Example 2: transpose (i, j) → (j, i)
C = torch.randn(4, 3)
C_T = torch.einsum('ij->ji', C)
print(f"\ntranspose: C({C.shape}) → einsum('ij->ji', C) = {C_T.shape}")
print(f"  equal to C.T: {torch.allclose(C_T, C.T)}")

# Example 3: batched matrix multiplication (a common operation after mel_to_batch)
batch_A = torch.randn(2, 4, 3)  # batch=2, n=4, m=3
batch_B = torch.randn(2, 3, 5)  # batch=2, m=3, k=5
result_batchmm = torch.bmm(batch_A, batch_B)
result_batchein = torch.einsum('bnm,bmk->bnk', batch_A, batch_B)
print(f"\nbatched matrix multiplication: A({batch_A.shape}) @ B({batch_B.shape}) = {result_batchmm.shape}")
print(f"  einsum('bnm,bmk->bnk', A, B) gives {result_batchein.shape}")
print(f"  equal to the torch.bmm result: {torch.allclose(result_batchmm, result_batchein)}")

# Example 4: vector dot product (i,) · (i,) → () a scalar
v1 = torch.tensor([1.0, 2.0, 3.0])
v2 = torch.tensor([4.0, 5.0, 6.0])
dot_result = torch.einsum('i,i->', v1, v2)
print(f"\nvector dot product: v1({v1.shape}) · v2({v2.shape}) = {dot_result.item():.0f}")
print(f"  einsum('i,i->', v1, v2) gives a scalar")
print(f"  check: v1@v2={v1@v2:.0f}")

## 4. ✏️ Implement `mel_to_batch(mel_list)`

**Signature**:
```python
def mel_to_batch(mel_list: list[np.ndarray]) -> torch.Tensor:
    # mel_list: list of (T, n_mels) ndarray
    # returns: (B, 1, T, n_mels) float32 Tensor
```

**Why is the output format (B, 1, T, n_mels)?**

This is the standard input format of PyTorch's Conv2d, `(B, C, H, W)`:
- `B` = batch size, how many samples are processed at once
- `C` = channel; a color image has 3 RGB channels, while an MFCC is single-channel audio, so C=1
- `H, W` = height and width; for an MFCC we treat **time T as the height** and **frequency n_mels as the width** (by convention)

```python
MFCC features     standard CNN format     meaning of the dimensions
┌────────┐      ┌───────────────┐
│ (T×40) │ →    │ (B, 1, T, 40) │
│ MFCC   │      │ (batch, channel, time, freq)
│ each   │      └───────────────┘
│ row:   │
│ time   │
└────────┘
  freq→
```

Why this order? Plain historical habit — the `(B,C,H,W)` format used by image CNNs is the most efficient when designing convolution kernels. Audio uses the same framework, so it gets converted into the same format.

**A 4-step implementation route**:

| Step | Operation | Notes |
|---|---|---|
| 1 | `np.stack(mel_list)` | join into a `(B, T, n_mels)` ndarray |
| 2 | `torch.from_numpy(...).float()` | convert to a Tensor, dtype → float32 |
| 3 | `.unsqueeze(1)` | insert the channel dimension → `(B, 1, T, n_mels)` |
| 4 | return | check that `.shape == (B, 1, T, n_mels)` |

**About requires_grad**:

The output Tensor's `requires_grad` should be `False` (the default). The reason: MFCC features are the fixed output of feature extraction from audio (an intermediate product), and we don't need to differentiate through them. Gradients are only needed for the model's **parameters** (weights and biases), not for these input features. That saves memory and computation.

**Acceptance criteria**:
- input `[np.zeros((80, 40))] * 3` → output shape `(3, 1, 80, 40)`
- `dtype == torch.float32` (not float64)
- the return value is a `torch.Tensor` (not an ndarray)
- `requires_grad == False` (the default)

In [ ]:
def mel_to_batch(mel_list):
    """
    mel_list : list of np.ndarray, each shape (T, n_mels)
    returns  : torch.Tensor shape (B, 1, T, n_mels), dtype float32
    """
    raise NotImplementedError("TODO: stack mel_list → unsqueeze(1) → float32 Tensor")

In [ ]:
# Check mel_to_batch
batch = mel_to_batch([np.zeros((80, 40)) for _ in range(3)])
assert batch.shape == (3, 1, 80, 40), f'wrong shape: {batch.shape}'
assert batch.dtype == torch.float32,   f'wrong dtype: {batch.dtype}'
print('✅ mel_to_batch shape:', batch.shape, '  dtype:', batch.dtype)

## 5. Parameter experiment: `requires_grad` and gradient flow

Experiment settings and what to expect:
- `requires_grad=False` (the default): `.grad` stays `None`, and calling `.backward()` raises a `RuntimeError`
- `requires_grad=True`: after each `.backward()`, `.grad` **accumulates** (it isn't overwritten), so you need `x.grad.zero_()` between repeated calls
- the `torch.no_grad()` context: operations inside it don't build a computation graph, saving memory during inference

**Why do gradients accumulate by default?**

To support **mini-batch gradient accumulation**. When training large models, GPU memory can't hold a full batch, so it's split into several mini-batches:

```python
# pseudocode: training with gradient accumulation
optimizer.zero_grad()  # zero once

for mini_batch_idx in range(4):  # split the big batch into 4 small ones
    x, y = load_minibatch()
    loss = model(x, y)
    loss.backward()  # no zeroing each time → gradients accumulate
    # ↓ each mini-batch's gradient is added on top

optimizer.step()  # update the weights only once at the end, with the accumulated gradient
optimizer.zero_grad()  # only now zero, ready for the next big batch
```

If PyTorch zeroed by default, the code above would be impossible. So it's designed as **accumulate by default, zero by hand**. It really is a beginner's trap (see experiment B), but it's a necessary design.

Below we check the gradient-accumulation trap, the correct loop, and the effect of `no_grad`.

In [ ]:
# Experiment A: requires_grad=False gives no gradient
x_ng = torch.tensor(3.0)           # requires_grad defaults to False
y_ng = x_ng ** 2
print('requires_grad=False, x_ng.grad:', x_ng.grad)  # None

# Experiment B: the gradient-accumulation trap
x2 = torch.tensor(3.0, requires_grad=True)
for _ in range(3):
    y2 = x2 ** 2 + 2 * x2
    y2.backward()
    # without zero_(), the gradient accumulates
print('x2.grad after 3 backward calls:', x2.grad.item())  # 8*3 = 24 (accumulated)

# The right way: zero on every iteration
x3 = torch.tensor(3.0, requires_grad=True)
for _ in range(3):
    if x3.grad is not None:
        x3.grad.zero_()
    y3 = x3 ** 2 + 2 * x3
    y3.backward()
print('x3.grad each time, with zeroing:', x3.grad.item())        # 8.0 (correct)

# Experiment C: torch.no_grad() builds no graph
x4 = torch.tensor(3.0, requires_grad=True)
with torch.no_grad():
    y4 = x4 ** 2
print('y4.requires_grad under no_grad:', y4.requires_grad)  # False

In [ ]:
# Experiment D: gradient accumulation in practice (mini-batch gradient accumulation)
print("\nExperiment D: what gradient accumulation is actually for ← the mini-batch scenario")
print("-" * 50)

# Scenario: split one batch into 2 mini-batches (not enough GPU memory)
# the gradients of the two mini-batches should be combined (added together)

def compute_loss_on_minibatch(x, target_x):
    """Compute one mini-batch's loss: (x - target_x)^2"""
    return ((x - target_x) ** 2).sum()

x = torch.tensor(3.0, requires_grad=True)
# suppose the batch's goal is for x to approach 0
# we split it into two mini-batches and compute each one

# mini-batch 1
loss1 = compute_loss_on_minibatch(x, target_x=0.0)
loss1.backward()
print(f"after mini-batch 1 backward, x.grad = {x.grad.item():.1f}")  # 6.0

# mini-batch 2 (no zeroing, the gradient accumulates)
loss2 = compute_loss_on_minibatch(x, target_x=0.0)
loss2.backward()  # gradient accumulates!
print(f"after mini-batch 2 backward, x.grad = {x.grad.item():.1f}")  # 6.0 + 6.0 = 12.0

print(f"gradients of both mini-batches accumulated: x.grad = {x.grad.item():.1f}")
print("in real training, this is the moment to call optimizer.step() and update the weights")

# starting the next batch: zero first
x.grad.zero_()
print(f"after zeroing, x.grad = {x.grad}")

# Contrast: zeroing every time (the wrong way)
print("\n[Contrast] If you accidentally zero every time (wrong):")
x2 = torch.tensor(3.0, requires_grad=True)
for i in range(2):
    x2.grad.zero_()  # ← you shouldn't zero after every mini-batch!
    loss = compute_loss_on_minibatch(x2, target_x=0.0)
    loss.backward()
    print(f"after mini-batch {i+1}, x.grad = {x2.grad.item():.1f}")
print("result: the two mini-batches' gradients overwrite each other, and in the end only one mini-batch's gradient is used (wrong!)")

## Wrap-up

`mel_to_batch()` turns a list of numpy MFCCs into a `float32` Tensor of shape `(B, 1, T, n_mels)`, the bridge between Aurora's Month 2 dataset loader and the CNN acoustic model. `torch.from_numpy` handles the zero-copy conversion and `unsqueeze(1)` adds the channel dimension; together they deliver the audio features in the `B,C,H,W` format that `Conv2d` needs. The `requires_grad` mechanism makes a Tensor build the computation graph as the forward pass runs, and `backward()` does the whole backward differentiation in one call. The next lesson (L60) takes PyTorch's autograd apart: tracing `grad_fn`, understanding `retain_grad` and gradient accumulation, and checking PyTorch's gradients number by number against the `Value` engine you hand-wrote in L56.

## ✏️ Whiteboard challenge: work out the core Tensor concepts by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: What's the difference between `torch.from_numpy(arr)` and `torch.tensor(arr)`? Which saves more memory?
(from_numpy shares memory (zero-copy), so changing arr changes the Tensor; tensor() copies the data — safer, but uses more memory)

**Question 2**: For a Tensor `x` with `requires_grad=True`, what is `x.grad` after `y = x**2; y.backward()`?
(`dy/dx = 2x`; if x=3.0 then x.grad=6.0)

**Question 3**: What's the difference between `view(-1, 4)` and `reshape(-1, 4)`? Which is safer?
(view needs contiguous memory and raises an error otherwise; reshape uses a view when it can and copies otherwise — safer)

**Question 4**: To feed MFCC features `(T, 40)` into a 2D CNN, which two shape transformations do they go through?
(step 1: stack B of them → (B,T,40); step 2: unsqueeze(1) → (B,1,T,40), inserting the channel dimension)

**Question 5**: Why does PyTorch default to `float32` while NumPy defaults to `float64`?
(float32 is 2–8× as fast as float64 on a GPU, and precise enough for deep learning; NumPy is aimed at scientific computing, where float64 is the standard)

When you've finished, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import torch, numpy as np

# Q1: from_numpy shares memory
arr_q = np.array([1.0, 2.0, 3.0])
t_q = torch.from_numpy(arr_q)
arr_q[0] = 99.0
assert t_q[0].item() == 99.0, "from_numpy should share memory"
print(f"Q1 ✅  from_numpy shares memory: set arr[0]=99 → t[0]={t_q[0].item():.0f}")

# Q2: requires_grad backward
x_q = torch.tensor(3.0, requires_grad=True)
y_q = x_q ** 2
y_q.backward()
assert abs(x_q.grad.item() - 6.0) < 1e-6, f"x.grad={x_q.grad.item()}"
print(f"Q2 ✅  x=3.0, y=x²: dy/dx={x_q.grad.item():.1f} = 2×3 = 6")

# Q3: view vs reshape
a_q = torch.arange(12).float()
b_view = a_q.view(3, 4)
b_reshape = a_q.reshape(3, 4)
assert b_view.shape == b_reshape.shape == (3, 4)
print(f"Q3 ✅  view and reshape both give (3,4); view needs contiguous memory, reshape is more forgiving")

# Q4: mel_to_batch shape transformation
try:
    mel_q = [np.zeros((80, 40)) for _ in range(3)]
    batch_q = mel_to_batch(mel_q)
    assert batch_q.shape == (3, 1, 80, 40), f"shape={batch_q.shape}"
    assert batch_q.dtype == torch.float32
    print(f"Q4 ✅  mel_to_batch: (80,40)×3 → {tuple(batch_q.shape)}, dtype={batch_q.dtype}")
except (NotImplementedError, TypeError):
    print(f"Q4 ✅  mel_to_batch: stack→(3,80,40)→from_numpy→float()→unsqueeze(1)→(3,1,80,40) (to be implemented)")

# Q5: float32 vs float64 (concept check)
t32 = torch.zeros(1000, dtype=torch.float32)
t64 = torch.zeros(1000, dtype=torch.float64)
assert t32.element_size() == 4
assert t64.element_size() == 8
print(f"Q5 ✅  float32={t32.element_size()} bytes/element, float64={t64.element_size()} bytes, GPU speed ratio ≈ 2-8×")
print("\n🎉 Tensor basics whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l59_review = {
    "tensor_vs_ndarray":       None,  # remember the two things a Tensor adds over an ndarray, GPU + gradients? True/False
    "from_numpy_shared_mem":   None,  # understand that from_numpy shares memory and tensor() copies? True/False
    "requires_grad_backward":  None,  # can you work out the autograd result of a simple function by hand? True/False
    "mel_to_batch_impl":       None,  # mel_to_batch() implemented correctly, shape/dtype checks pass? True/False
    "whiteboard_passed":       None,  # all 5 whiteboard questions done? True/False
}

unfilled = [k for k, v in l59_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l59_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L59 fully cleared! On to L60: how autograd works')

---

→ **Next lesson** · [L60 · How autograd works](L60_pytorch_autograd.ipynb)

> Next lesson covers **how autograd works**: `grad_fn` graphs, `backward()`, `retain_grad` and gradient accumulation, with a numerical comparison against your hand-written `Value` engine.